In [ ]:
import numpy as np
import pandas as pd
import pulp
import matplotlib.pyplot as plt

In [ ]:
products = ["Table", "Chair"]
prices = [30, 10]
woodworking_time = [6, 3]
storage_space = [4, 1]

In [ ]:
def solve_carpenter(products, prices, woodworking_time, storage_space, cat="Continuous"):
    prices = np.asarray(prices, dtype=float)
    woodworking_time = np.asarray(woodworking_time, dtype=float)
    storage_space = np.asarray(storage_space, dtype=float)

    model = pulp.LpProblem("Carpenter", pulp.LpMaximize)

    x = model.add_variable_dict("x", products, lowBound=0, cat=cat)

    model += pulp.lpSum(prices[i] * x[s] for i, s in enumerate(products))

    # constraints
    model += pulp.lpSum(woodworking_time[i] * x[s] for i, s in enumerate(products)) <= 40, "Man_hours"
    model += x[products[1]] >= 3 * x[products[0]], "Demand"
    model += pulp.lpSum(storage_space[i] * x[s] for i, s in enumerate(products)) <= 16, "Storage"

    result = model.solve(pulp.HiGHS(msg=False))
    return model, x, result

In [ ]:
model, x, result = solve_carpenter(products, prices, woodworking_time, storage_space)
if result.status_str == "Optimal":
    print("Optimal Production Plan:")
    for s in products:
        print(f"{s}: {x[s].value():.2f} units")
    print("Total Revenue:", round(result.objective, 2))

In [ ]:
x1 = np.linspace(0, 5, 400)

# Each constraint written as x2 = ... (its boundary line)
man_hours = (40 - 6 * x1) / 3     # 6*x1 + 3*x2 <= 40
storage = 16 - 4 * x1             # 4*x1 + x2 <= 16
demand = 3 * x1                   # x2 >= 3*x1

fig, ax = plt.subplots(figsize=(8, 6))

# 1. Constraint boundary lines
ax.plot(x1, man_hours, label=r"Man-hours: $6x_1 + 3x_2 \leq 40$")
ax.plot(x1, storage, label=r"Storage: $4x_1 + x_2 \leq 16$")
ax.plot(x1, demand, color="tab:green", label=r"Demand: $x_2 \geq 3x_1$")

# 2. Feasible region: above the demand line, below both man-hours and storage, x1 >= 0
x1_feasible = np.linspace(0, 16 / 7, 200)
lower = 3 * x1_feasible
upper = np.minimum((40 - 6 * x1_feasible) / 3, 16 - 4 * x1_feasible)
ax.fill_between(x1_feasible, lower, upper, color="grey", alpha=0.3, label="Feasible region")

# 3. Corner points of the feasible region and the revenue at each
corners = [(0, 0), (0, 40 / 3), (4 / 3, 32 / 3), (16 / 7, 48 / 7)]
for cx, cy in corners:
    z = 30 * cx + 10 * cy
    ax.plot(cx, cy, "ko", clip_on=False)
    ax.annotate(f"({cx:.2f}, {cy:.2f})\nz = {z:.2f}", (cx, cy), textcoords="offset points", xytext=(8, 8))

# 4. Optimum
ax.plot(4 / 3, 32 / 3, "r*", markersize=18, clip_on=False, label="Optimum (1.33, 10.67), z = 146.67")

ax.set_xlim(0, 5)
ax.set_ylim(0, 18)
ax.set_xlabel("$x_1$ (tables per week)")
ax.set_ylabel("$x_2$ (chairs per week)")
ax.set_title("The Good Carpenter - graphical solution")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.show()